## Cleaning & standardizing — nulls, types, casting

Two jobs, run together on almost every silver build: get rid of bad nulls, then pin real types.

**Cleaning nulls** — three operations cover ninety percent of the work:

```python
from pyspark.sql.functions import when, col

# Drop rows with nulls in required columns
clean = df.dropna(subset=["transaction_id", "customer_id", "amount"])

# Replace nulls with a default
clean = clean.fillna({"merchant_category": "unknown", "is_flagged": False})

# Conditional replacement
clean = clean.withColumn(
    "status",
    when(col("status").isNull(), "pending").otherwise(col("status")),
)
```

**The `dropna` tip the exam likes:** `dropna(how="any")` (the default) drops a row if *any* column is null; `dropna(how="all")` drops only when *every* column is null. Scope it with `subset=`.

**Standardising types** — bronze lands everything as `STRING` on purpose, so schema-on-read never loses data. Silver pins the real types:

```python
from pyspark.sql.functions import to_timestamp

silver = (df
    .withColumn("amount",         col("amount").cast("decimal(18,2)"))
    .withColumn("is_flagged",     col("is_flagged").cast("boolean"))
    .withColumn("transaction_at", to_timestamp("transaction_at")))
```

**The casting pitfall:** a string that won't parse becomes `NULL` **silently — not an error.** Defend against it: land bad rows in a quarantine table and alert on row count, or use Spark SQL's `try_cast`, which is `CAST` but explicit about its null-on-failure behaviour.
